# 02 - Data Preprocessing

Clean and normalize the multi-modal dataset scraped from the web.
This notebook prepares the data for the 5 different models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import os
import pickle

os.makedirs('dataset/processed', exist_ok=True)

print('Directories created')

I0000 00:00:1775600886.408935   78763 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775600886.488760   78763 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775600888.293294   78763 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Directories created


## Load Scraped Data

In [2]:
# Load data
df_restaurants = pd.read_csv('dataset/restaurant_tabular_data.csv')
df_reviews = pd.read_csv('dataset/restaurant_reviews.csv')
df_images = pd.read_csv('dataset/restaurant_images.csv')

print(f'Loaded {len(df_restaurants)} restaurants')
print(f'Loaded {len(df_reviews)} reviews')
print(f'Loaded {len(df_images)} image references')

Loaded 150 restaurants
Loaded 150 reviews
Loaded 150 image references


## Preprocess Tabular Data

In [3]:
# Encode categorical variables
price_encoder = LabelEncoder()
cuisine_encoder = LabelEncoder()
neighborhood_encoder = LabelEncoder()

df_restaurants['price_encoded'] = price_encoder.fit_transform(df_restaurants['price_range'])
df_restaurants['cuisine_encoded'] = cuisine_encoder.fit_transform(df_restaurants['cuisine_type'])
df_restaurants['neighborhood_encoded'] = neighborhood_encoder.fit_transform(df_restaurants['neighborhood'])

# Normalize numerical features
scaler = StandardScaler()
df_restaurants[['rating_scaled', 'review_count_scaled']] = scaler.fit_transform(
    df_restaurants[['rating', 'review_count']]
)

print('Preprocessed tabular data')

Preprocessed tabular data


In [4]:
# Prepare tabular features (NO data leakage: rating_scaled is NOT included)
tabular_features = df_restaurants[['price_encoded', 'cuisine_encoded', 'neighborhood_encoded', 'review_count_scaled']].values
tabular_target = df_restaurants['rating'].values

# Split tabular data
X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    tabular_features, tabular_target, test_size=0.2, random_state=42
)

print(f'Tabular train/test split: {X_tab_train.shape}, {X_tab_test.shape}')

Tabular train/test split: (120, 5), (30, 5)


## Preprocess Text Data

In [5]:
# Tokenize text
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df_reviews['review_text'])

sequences = tokenizer.texts_to_sequences(df_reviews['review_text'])
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Encode sentiment labels
sentiment_encoder = LabelEncoder()
sentiment_labels = sentiment_encoder.fit_transform(df_reviews['sentiment'])

# Split text data
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    padded_sequences, sentiment_labels, test_size=0.2, random_state=42
)

print(f'Text train/test split: {X_text_train.shape}, {X_text_test.shape}')

Text train/test split: (120, 100), (30, 100)


## Preprocess Image Data

In [6]:
# Prepare image labels
image_labels = df_restaurants['is_high_tier'].values

# Load and resize images
img_height, img_width = 224, 224

def load_and_preprocess_images(image_paths, target_size=(img_height, img_width)):
    """
    Load and preprocess images from file paths
    """
    images = []
    for path in image_paths:
        try:
            if os.path.exists(path):
                img = load_img(path, target_size=target_size)
                img_array = img_to_array(img)
                img_array = img_array / 255.0  # Normalize to [0, 1]
                images.append(img_array)
            else:
                # If image doesn't exist, create a placeholder
                images.append(np.random.rand(*target_size, 3))
        except Exception as e:
            # If loading fails, create a placeholder
            images.append(np.random.rand(*target_size, 3))
    
    return np.array(images)

In [7]:
# Get image paths
image_paths = df_images['local_path'].values

# Load images
print('Loading images...')
all_images = load_and_preprocess_images(image_paths)
print(f'Loaded {len(all_images)} images')

Loading images...
Loaded 150 images


In [8]:
# Split image data
X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    all_images, image_labels, test_size=0.2, random_state=42
)

print(f'Image train/test split: {X_img_train.shape}, {X_img_test.shape}')

Image train/test split: (120, 224, 224, 3), (30, 224, 224, 3)


## Save All Processed Data

In [9]:
# Save tabular data
np.save('dataset/processed/X_tab_train.npy', X_tab_train)
np.save('dataset/processed/X_tab_test.npy', X_tab_test)
np.save('dataset/processed/y_tab_train.npy', y_tab_train)
np.save('dataset/processed/y_tab_test.npy', y_tab_test)

print('Saved tabular data')

Saved tabular data


In [10]:
# Save text data
np.save('dataset/processed/X_text_train.npy', X_text_train)
np.save('dataset/processed/X_text_test.npy', X_text_test)
np.save('dataset/processed/y_text_train.npy', y_text_train)
np.save('dataset/processed/y_text_test.npy', y_text_test)

print('Saved text data')

Saved text data


In [11]:
# Save image data
np.save('dataset/processed/X_img_train.npy', X_img_train)
np.save('dataset/processed/X_img_test.npy', X_img_test)
np.save('dataset/processed/y_img_train.npy', y_img_train)
np.save('dataset/processed/y_img_test.npy', y_img_test)

print('Saved image data')

Saved image data


In [12]:
# Save encoders
with open('dataset/processed/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('dataset/processed/sentiment_encoder.pkl', 'wb') as f:
    pickle.dump(sentiment_encoder, f)
with open('dataset/processed/price_encoder.pkl', 'wb') as f:
    pickle.dump(price_encoder, f)
with open('dataset/processed/cuisine_encoder.pkl', 'wb') as f:
    pickle.dump(cuisine_encoder, f)
with open('dataset/processed/neighborhood_encoder.pkl', 'wb') as f:
    pickle.dump(neighborhood_encoder, f)

print('Saved encoders')

Saved encoders


## Summary

Data preprocessing complete:
- Tabular data: Encoded and normalized, split into train/test
- Text data: Tokenized and padded, split into train/test
- Image data: Loaded, resized, normalized, split into train/test
- All encoders saved for later use

All data is ready for the 5 models.

**Data Leakage Fix:** rating_scaled has been REMOVED from tabular input features.
Tabular features now include ONLY: price_encoded, cuisine_encoded, neighborhood_encoded, review_count_scaled.
The target variable (rating) is no longer included as an input, ensuring valid model results.